# Playing with `pyphi.serialize`

The new `msgspec`-based serializer for PyPhi results. One API, two wire
formats (JSON + msgpack) from a single typed schema, with a **normalized**
cause-effect structure: distinctions are stored once and relations reference
them by index instead of embedding each distinction many times over.

In [1]:
import pyphi
from pyphi import examples
from pyphi import serialize

pyphi.config.progress_bars = False  # quiet output for the demo

system = examples.rule110_system()
sia = system.sia()  # system irreducibility analysis (Φ_s)
ces = system.ces()  # full cause-effect structure (distinctions + relations)
print(f"Φ_s = {float(sia.phi):.4f}   |   {len(ces.distinctions)} distinctions")

/Users/will/projects/pyphi/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Φ_s = 2.0000   |   4 distinctions


## Serialize to JSON or msgpack

`dumps` returns `bytes`. `format=` is `"json"` (default, human-readable) or
`"msgpack"` (compact binary).

In [2]:
js = serialize.dumps(ces, format="json")
mp = serialize.dumps(ces, format="msgpack")

print(f"JSON    : {len(js):>6,} bytes")
print(f"msgpack : {len(mp):>6,} bytes")
print()
print("first 400 bytes of the JSON:")
print(js[:400].decode())
print("first 400 bytes of the msgpack:")
print(mp[:400])

JSON    : 63,454 bytes
msgpack : 48,886 bytes

first 400 bytes of the JSON:
{"format_version":1,"payload":{"type":"ces","sia":{"type":"iit4_sia","phi":{"type":"pyphi_float","value":2.0},"partition":{"type":"directed_set_partition","node_indices":[0,1,2],"cut_matrix":"k05VTVBZAQB2AHsnZGVzY3InOiAnPGk4JywgJ2ZvcnRyYW5fb3JkZXInOiBGYWxzZSwgJ3NoYXBlJzogKDMsIDMpLCB9ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIAoAAAAAAAAAAAEAAAAAAAAAAAAAAAAAAAABAAAA
first 400 bytes of the msgpack:
b"\x82\xaeformat_version\x01\xa7payload\x84\xa4type\xa3ces\xa3sia\xde\x00\x11\xa4type\xa8iit4_sia\xa3phi\x82\xa4type\xabpyphi_float\xa5value\xcb@\x00\x00\x00\x00\x00\x00\x00\xa9partition\x85\xa4type\xb6directed_set_partition\xacnode_indices\x93\x00\x01\x02\xaacut_matrix\xc4\xc8\x93NUMPY\x01\x00v\x00{'descr': '<i8', 'fortran_order': False, 'shape': (3, 3), }                                                          \n\x00\x00\x00\x00\x00\x00\x00\x00\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00

## Round-trip: decode back to a live object

`loads` reconstructs the original PyPhi object. Equality is value-based, so a
round-trip compares equal to the original.

In [3]:
restored = serialize.loads(js, format="json")
restored_mp = serialize.loads(mp, format="msgpack")

print("type preserved :", type(restored).__name__)
print("equal (json)   :", restored == ces)
print("equal (msgpack):", restored_mp == ces)
print("Φ_s round-trips :", float(restored.sia.phi))

type preserved : CauseEffectStructure
equal (json)   : True
equal (msgpack): True
Φ_s round-trips : 2.0


## The normalized cause-effect structure

A relation is a set of distinctions. The old serializer embedded each
distinction *in full* inside every relation it belonged to — highly redundant.
The new format stores the distinctions **once** in a table and gives each
relation only the integer indices of its members.

In [4]:
import json

doc = json.loads(serialize.dumps(ces, format="json"))
payload = doc["payload"]
print("document keys :", list(doc.keys()))
print("format_version:", doc["format_version"])
print("payload type  :", payload["type"])
print(
    "distinctions  :", len(payload["distinctions"]["concepts"]), "stored once in a table"
)
rels = payload["relations"]
print("relations type:", rels["type"])
if rels.get("relations"):
    print("a relation is just indices:", rels["relations"][0]["distinction_indices"])

document keys : ['format_version', 'payload']
format_version: 1
payload type  : ces
distinctions  : 4 stored once in a table
relations type: concrete_relations_ref
a relation is just indices: [0, 1]


In [5]:
rels

{'type': 'concrete_relations_ref',
 'relations': [{'type': 'relation_ref', 'distinction_indices': [0, 1]},
  {'type': 'relation_ref', 'distinction_indices': [0, 1, 2]},
  {'type': 'relation_ref', 'distinction_indices': [0, 2, 3]},
  {'type': 'relation_ref', 'distinction_indices': [0, 3]},
  {'type': 'relation_ref', 'distinction_indices': [1, 2]},
  {'type': 'relation_ref', 'distinction_indices': [2, 3]},
  {'type': 'relation_ref', 'distinction_indices': [0, 2]},
  {'type': 'relation_ref', 'distinction_indices': [0, 1, 2, 3]},
  {'type': 'relation_ref', 'distinction_indices': [1, 2, 3]},
  {'type': 'relation_ref', 'distinction_indices': [1, 3]},
  {'type': 'relation_ref', 'distinction_indices': [0, 1, 3]}]}

## Write to / read from a file

In [6]:
from pathlib import Path

# save() takes a path (or a binary file object) and infers the wire format
# from the extension: .json -> JSON, .msgpack / .mpk -> msgpack.
serialize.save(ces, "ces.json")
serialize.save(ces, "ces.msgpack")

# load() infers the format the same way.
loaded = serialize.load("ces.json")

p_json, p_mp = Path("ces.json"), Path("ces.msgpack")
print("loaded from file equals original:", loaded == ces)
print(
    f"on disk: {p_json.name} = {p_json.stat().st_size:,} B   "
    f"{p_mp.name} = {p_mp.stat().st_size:,} B"
)

loaded from file equals original: True
on disk: ces.json = 63,454 B   ces.msgpack = 48,886 B


## Try it on other result types

Every PyPhi result type round-trips — swap `ces` above for any of these.

In [7]:
from pyphi import actual
from pyphi.actual import Transition

ac_sub = examples.actual_causation_substrate()
transition = Transition(
    ac_sub,
    before_state=(1, 1),
    after_state=(1, 1),
    cause_indices=(0, 1),
    effect_indices=(0, 1),
)
account = actual.account(transition)

for obj in [system.substrate, system, sia, transition, account]:
    rt = serialize.loads(serialize.dumps(obj, format="msgpack"), format="msgpack")
    print(f"{type(obj).__name__:<28} round-trips: {rt == obj}")

Substrate                    round-trips: True
System                       round-trips: True
SystemIrreducibilityAnalysis round-trips: True
Transition                   round-trips: True
Account                      round-trips: True


---

**Notes**

- numpy arrays (repertoires, TPMs) are stored as their exact `.npy` bytes —
  base64 in JSON, raw in msgpack — and loaded with `allow_pickle=False`.
- The document carries a single top-level `format_version`.
- This serializer was validated to reproduce the trusted legacy goldens exactly.